In [ ]:
import sys
from urllib.parse import quote
from SPARQLWrapper import SPARQLWrapper, JSON
from IPython.display import IFrame, display, HTML

endpoint_url = "https://query.wikidata.org/sparql"

query = """#title: Visual Timeline of exhibitions at the Sprengel Museum
#defaultView:Timeline
SELECT ?exhibition ?exhibitionLabel ?date ?image WHERE {
  ?exhibition wdt:P31/wdt:P279* wd:Q464980; # Instance of or subclass of exhibition
              wdt:P276 wd:Q510144.          # Location: Sprengel Museum
  OPTIONAL { ?exhibition wdt:P571 ?date. }
  OPTIONAL { ?exhibition wdt:P580 ?date. }
  OPTIONAL { ?exhibition wdt:P18 ?image. }
  FILTER(BOUND(?date))
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en,de". }
}
ORDER BY ?date"""


def get_results(endpoint_url, query):
    user_agent = "WDQS-example Python/%s.%s" % (sys.version_info[0], sys.version_info[1])
    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()


results = get_results(endpoint_url, query)
bindings = results["results"]["bindings"]
count_with_date = sum(1 for result in bindings if "date" in result)

display(
    HTML(
        f"""
<style>
  .notebook-toolbar {{
    margin: 10px 0 14px;
    padding: 10px 12px;
    display: flex;
    align-items: center;
    gap: 12px;
    background: #f5f7fa;
    border: 1px solid #d0d7de;
    border-radius: 8px;
    font-family: Arial, sans-serif;
    font-size: 14px;
  }}

  .notebook-toolbar button {{
    border: 1px solid #c7ced6;
    background: #ffffff;
    color: #24292f;
    border-radius: 6px;
    padding: 6px 10px;
    cursor: pointer;
    font-size: 13px;
  }}

  .notebook-toolbar button:hover {{
    background: #f3f4f6;
  }}

  .notebook-toolbar a {{
    color: #0969da;
    text-decoration: none;
    font-weight: 500;
  }}

  .notebook-toolbar a:hover {{
    text-decoration: underline;
  }}

  body.hide-code .jp-Cell-inputWrapper {{
    display: none;
  }}
</style>

<div class='notebook-toolbar'>
  <button id='toggle-code' type='button'>Show code</button>
  <a href='notebook.ipynb' target='_blank' rel='noopener noreferrer'>Open notebook file (.ipynb)</a>
</div>

<div style='font-family: Arial, sans-serif; margin: 10px 0 14px;'>
  <div style='font-size: 16px; font-weight: 600; color: #1f2937;'>Sprengel Museum Timeline</div>
  <div style='font-size: 13px; color: #6b7280;'>Exhibitions with date: {count_with_date}</div>
</div>

<script>
  (function () {{
    const button = document.getElementById('toggle-code');
    if (!button) return;

    document.body.classList.add('hide-code');

    button.addEventListener('click', function () {{
      document.body.classList.toggle('hide-code');
      button.textContent = document.body.classList.contains('hide-code') ? 'Show code' : 'Hide code';
    }});
  }})();
</script>
"""
    )
)

embed_url = "https://query.wikidata.org/embed.html#" + quote(query)
display(IFrame(src=embed_url, width="100%", height=520))